# Loading

In [1]:
import pandas as pd

# Population counts by boundary (from Boundaries_2023_Analysis.ipynb)
ward_pop = pd.read_csv('ward_pop.csv')
township_pop = pd.read_csv('township_pop.csv')

# Turnout by boundary (from Election_Data_2023.ipynb and Suburban_Cook_Analysis.ipynb)
ward_election = pd.read_csv('ward_election_clean.csv')
suburban_election = pd.read_csv('suburban_election_clean.csv')

print(ward_pop.shape, ward_election.shape)
print(township_pop.shape, suburban_election.shape)

(50, 4) (50, 4)
(30, 4) (30, 4)


#  Cook County Gap Metric

In [2]:
ward_election = ward_election.rename(columns={'ward': 'Ward'})

ward_combined = ward_pop.merge(ward_election, on='Ward', how='inner')

township_combined = township_pop.merge(suburban_election, on='Township_Name', how='inner')

print(ward_combined.shape)   
print(ward_combined.head())
print(township_combined.shape) 
print(township_combined.head())

(50, 7)
   Ward  muslim_foreign_born  naturalized_citizens  non_citizens  registered  \
0   1.0                  323                  3624          3085       36526   
1   2.0                  405                  3108          3561       31790   
2   3.0                  884                  3050          2794       33043   
3   4.0                  890                  3535          5637       30722   
4   5.0                  353                  1180          2920       26941   

   total    turnout  
0  21094  57.750643  
1  18918  59.509280  
2  15768  47.719638  
3  16457  53.567476  
4  14609  54.225901  
(30, 7)
  Township_Name  muslim_foreign_born  naturalized_citizens  non_citizens  \
0    BARRINGTON                  206                  2657          1139   
1        BERWYN                  413                  7092          7124   
2         BLOOM                   52                  3802          3668   
3        BREMEN                  807                  5598         

In [3]:
# Using min-max, not percentile-rank, per supervisor's preference to preserve magnitude differences (percentile-rank flattens them)

min_pop = ward_combined['muslim_foreign_born'].min()
max_pop = ward_combined['muslim_foreign_born'].max()
min_turnout = ward_combined['turnout'].min()
max_turnout = ward_combined['turnout'].max()

In [4]:
ward_combined['concentration_norm'] = (ward_combined['muslim_foreign_born'] - min_pop) / (max_pop - min_pop)
ward_combined['turnout_norm'] = (ward_combined['turnout'] - min_turnout) / (max_turnout - min_turnout)
ward_combined['gap_score'] = ward_combined['concentration_norm'] - ward_combined['turnout_norm']

ward_ranked = ward_combined.sort_values('gap_score', ascending=False).reset_index(drop=True)
print(ward_ranked[['Ward', 'muslim_foreign_born', 'turnout', 'gap_score']].head(15))

    Ward  muslim_foreign_born    turnout  gap_score
0   50.0                 3888  42.573624   0.614321
1   14.0                  138  27.306808   0.016673
2   12.0                  345  29.792547   0.010182
3   15.0                    0  26.523559   0.000000
4   22.0                    0  27.976861  -0.034922
5   16.0                    0  28.444041  -0.046149
6   40.0                 2390  54.086840  -0.047626
7   28.0                  417  32.968978  -0.047629
8   31.0                  277  31.609346  -0.050965
9   10.0                  171  30.514149  -0.051911
10  37.0                   31  31.740716  -0.117394
11  24.0                  133  32.944153  -0.120077
12  17.0                    0  34.875369  -0.200692
13  39.0                 1526  51.648384  -0.211253
14  20.0                   68  36.929856  -0.232571


In [5]:
min_pop_t = township_combined['muslim_foreign_born'].min()
max_pop_t = township_combined['muslim_foreign_born'].max()
min_turnout_t = township_combined['turnout'].min()
max_turnout_t = township_combined['turnout'].max()

In [6]:
# Checking how many areas fall below the population floor before deciding on a cutoff

township_combined['concentration_norm'] = (township_combined['muslim_foreign_born'] - min_pop_t) / (max_pop_t - min_pop_t)
township_combined['turnout_norm'] = (township_combined['turnout'] - min_turnout_t) / (max_turnout_t - min_turnout_t)
township_combined['gap_score'] = township_combined['concentration_norm'] - township_combined['turnout_norm']

township_ranked = township_combined.sort_values('gap_score', ascending=False).reset_index(drop=True)
print(township_ranked[['Township_Name', 'muslim_foreign_born', 'turnout', 'gap_score']].head(15))

   Township_Name  muslim_foreign_born    turnout  gap_score
0          NILES                 9307  43.282289   0.477710
1         CICERO                  259  25.377634   0.027097
2          MAINE                 5439  45.009469   0.011414
3        HANOVER                 2013  36.223054  -0.100669
4       STICKNEY                  531  33.048680  -0.167425
5          WORTH                 3286  43.633388  -0.179951
6         BERWYN                  413  34.347354  -0.217996
7        CALUMET                    7  33.345943  -0.232440
8         LEYDEN                  678  36.227230  -0.244339
9       THORNTON                  151  35.352924  -0.275501
10    SCHAUMBURG                 2247  43.181176  -0.278480
11        BREMEN                  807  39.531352  -0.326851
12         BLOOM                   52  37.608256  -0.351936
13         PALOS                 2027  46.262371  -0.392016
14       PROVISO                  545  43.146289  -0.460473


In [7]:
print((ward_ranked['muslim_foreign_born'] < 100).sum(), "of", len(ward_ranked), "wards below floor")
print((township_ranked['muslim_foreign_born'] < 100).sum(), "of", len(township_ranked), "townships below floor")

17 of 50 wards below floor
5 of 30 townships below floor


# Tier System

In [8]:
# Looking for a natural break in gap_score to justify where High tier starts

ward_sorted = ward_ranked.sort_values('gap_score', ascending=False).reset_index(drop=True)
ward_sorted['score_drop'] = ward_sorted['gap_score'].diff(-1)  

print(ward_sorted[['Ward', 'muslim_foreign_born', 'gap_score', 'score_drop']].head(51))

    Ward  muslim_foreign_born  gap_score  score_drop
0   50.0                 3888   0.614321    0.597649
1   14.0                  138   0.016673    0.006491
2   12.0                  345   0.010182    0.010182
3   15.0                    0   0.000000    0.034922
4   22.0                    0  -0.034922    0.011226
5   16.0                    0  -0.046149    0.001478
6   40.0                 2390  -0.047626    0.000002
7   28.0                  417  -0.047629    0.003337
8   31.0                  277  -0.050965    0.000946
9   10.0                  171  -0.051911    0.065482
10  37.0                   31  -0.117394    0.002684
11  24.0                  133  -0.120077    0.080614
12  17.0                    0  -0.200692    0.010561
13  39.0                 1526  -0.211253    0.021318
14  20.0                   68  -0.232571    0.004110
15  11.0                  562  -0.236681    0.001534
16  26.0                  118  -0.238215    0.035880
17   9.0                    0  -0.274094    0.

In [9]:
township_sorted = township_ranked.sort_values('gap_score', ascending=False).reset_index(drop=True)
township_sorted['score_drop'] = township_sorted['gap_score'].diff(-1)

print(township_sorted[['Township_Name', 'muslim_foreign_born', 'gap_score', 'score_drop']].head(30))

   Township_Name  muslim_foreign_born  gap_score  score_drop
0          NILES                 9307   0.477710    0.450614
1         CICERO                  259   0.027097    0.015683
2          MAINE                 5439   0.011414    0.112082
3        HANOVER                 2013  -0.100669    0.066756
4       STICKNEY                  531  -0.167425    0.012526
5          WORTH                 3286  -0.179951    0.038046
6         BERWYN                  413  -0.217996    0.014444
7        CALUMET                    7  -0.232440    0.011898
8         LEYDEN                  678  -0.244339    0.031163
9       THORNTON                  151  -0.275501    0.002978
10    SCHAUMBURG                 2247  -0.278480    0.048371
11        BREMEN                  807  -0.326851    0.025085
12         BLOOM                   52  -0.351936    0.040080
13         PALOS                 2027  -0.392016    0.068457
14       PROVISO                  545  -0.460473    0.043342
15        ORLAND        

In [10]:
# High = positive gap_score, a genuine natural break confirmed empirically (not arbitrary). Medium/Low split at the median of what's left, this split is organizational, not data-driven, since no clean break exists there

ward_ranked['tier'] = 'Below Threshold'

above_floor = ward_ranked['muslim_foreign_born'] >= 100
ward_ranked.loc[above_floor & (ward_ranked['gap_score'] > 0), 'tier'] = 'High'

remaining = above_floor & (ward_ranked['gap_score'] <= 0)
median_score = ward_ranked.loc[remaining, 'gap_score'].median()

ward_ranked.loc[remaining & (ward_ranked['gap_score'] >= median_score), 'tier'] = 'Medium'
ward_ranked.loc[remaining & (ward_ranked['gap_score'] < median_score), 'tier'] = 'Low'

print(ward_ranked['tier'].value_counts())

tier
Below Threshold    17
Medium             15
Low                15
High                3
Name: count, dtype: int64


In [11]:
township_ranked['tier'] = 'Below Threshold' 

above_floor = township_ranked['muslim_foreign_born'] >= 100


township_ranked.loc[above_floor & (township_ranked['gap_score'] > 0), 'tier'] = 'High'


remaining = above_floor & (township_ranked['gap_score'] <= 0)
median_score = township_ranked.loc[remaining, 'gap_score'].median()

township_ranked.loc[remaining & (township_ranked['gap_score'] >= median_score), 'tier'] = 'Medium'
township_ranked.loc[remaining & (township_ranked['gap_score'] < median_score), 'tier'] = 'Low'

print(township_ranked['tier'].value_counts())  

tier
Medium             11
Low                11
Below Threshold     5
High                3
Name: count, dtype: int64


In [12]:
ward_ranked.to_csv('ward_ranked.csv', index=False)
township_ranked.to_csv('township_ranked.csv', index=False)

In [13]:
wards_shapes = pd.read_csv('wards_shapes.csv')  
wards_shapes['Ward'] = wards_shapes['Ward'].astype(int)
ward_ranked['Ward'] = ward_ranked['Ward'].astype(int)

merged = wards_shapes.merge(ward_ranked, on='Ward', how='left')
merged.to_csv('ward_map_ready.csv', index=False)

# DuPage Gap Metric

In [14]:
dupage_full = pd.read_csv('dupage_township_pop.csv')

# Min-max normalize Muslim concentration and turnout (0-1 scale)
dupage_full['concentration_norm'] = (dupage_full['muslim_foreign_born'] - dupage_full['muslim_foreign_born'].min()) / \
                                      (dupage_full['muslim_foreign_born'].max() - dupage_full['muslim_foreign_born'].min())

dupage_full['turnout_norm'] = (dupage_full['turnout'] - dupage_full['turnout'].min()) / \
                                (dupage_full['turnout'].max() - dupage_full['turnout'].min())

# Gap score: high concentration + low turnout = high priority
dupage_full['gap_score'] = dupage_full['concentration_norm'] - dupage_full['turnout_norm']

print(dupage_full[['TownshipName', 'muslim_foreign_born', 'turnout', 'concentration_norm', 'turnout_norm', 'gap_score']].sort_values('gap_score', ascending=False))

    TownshipName  muslim_foreign_born    turnout  concentration_norm  \
1   Bloomingdale                 2878  46.856669            1.000000   
0        Addison                  707  47.394437            0.137465   
2  Downers Grove                 2809  60.414216            0.972586   
6          Wayne                 1260  51.568824            0.357171   
5     Naperville                 1222  52.413549            0.342074   
4         Milton                 2804  62.019079            0.970600   
8           York                 1977  58.415853            0.642034   
3          Lisle                 1588  58.784034            0.487485   
7       Winfield                  361  53.880014            0.000000   

   turnout_norm  gap_score  
1      0.000000   1.000000  
0      0.035467   0.101998  
2      0.894155   0.078431  
6      0.310779   0.046392  
5      0.366491  -0.024417  
4      1.000000  -0.029400  
8      0.762358  -0.120324  
3      0.786640  -0.299155  
7      0.463208  -

In [15]:
dupage_full.to_csv('dupage_township_pop_ranked.csv', index=False)
print('saved')

saved


In [16]:
# Merge DuPage shapes with ranked gap-score data, same pattern as ward_map_ready
dupage_shapes = pd.read_csv('dupage_shapes.csv')

dupage_shapes['TownshipName'] = dupage_shapes['TownshipName'].astype(str).str.strip()
dupage_full['TownshipName'] = dupage_full['TownshipName'].astype(str).str.strip()

dupage_map_ready = dupage_shapes.merge(dupage_full, on='TownshipName', how='left')
dupage_map_ready.to_csv('dupage_map_ready.csv', index=False)

print(dupage_map_ready.shape)
print(dupage_map_ready[['TownshipName', 'gap_score']].sort_values('gap_score', ascending=False))

(9, 12)
    TownshipName  gap_score
1   Bloomingdale   1.000000
0        Addison   0.101998
2  Downers Grove   0.078431
6          Wayne   0.046392
5     Naperville  -0.024417
4         Milton  -0.029400
8           York  -0.120324
3          Lisle  -0.299155
7       Winfield  -0.463208


# Combined Map

In [17]:
import pandas as pd

ward = pd.read_csv('ward_map_ready.csv')
cook_township = pd.read_csv('township_map_ready.csv')
dupage = pd.read_csv('dupage_map_ready.csv')

cols = ['name', 'geometry', 'gap_score', 'tier', 'muslim_foreign_born', 'naturalized_citizens', 'turnout', 'registered']

ward_std = ward.rename(columns={'Ward': 'name'})[cols]
ward_std['level'] = 'Ward'

cook_std = cook_township.rename(columns={'Township_Name': 'name'})[cols]
cook_std['level'] = 'Cook Township'

dupage_std = dupage.rename(columns={'TownshipName': 'name'})
dupage_std['tier'] = 'No Tier (DuPage)'
dupage_std = dupage_std[cols]
dupage_std['level'] = 'DuPage Township'

combined = pd.concat([ward_std, cook_std, dupage_std], ignore_index=True)
combined.to_csv('all_regions_map_ready.csv', index=False)
print(combined.shape)
print(combined['level'].value_counts())

(89, 9)
level
Ward               50
Cook Township      30
DuPage Township     9
Name: count, dtype: int64
